In [ ]:
%pip install neurokit2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 14.7 MB/s eta 0:00:00


In [ ]:
# import packages
import os
import pandas as pd
import numpy as np
import neurokit2 as nk


# Define a fuction to clean the raw signal

In [ ]:
# Function to clean ECG, RSP
def clean_ecg_rsp( sub_idx ):
    path = os.getcwd()
    # 4 difficult levels
    levels = [1, 2, 3, 4]
    # total 12 runs, 3 runs at each level
    for level in levels:
        if level == 1:
            runs = ['01', '07', '12']
        elif level == 2:
            runs = ['03', '08', '10']
        elif level == 3:
            runs = ['02', '05', '11'] if sub_idx != 30 else ['02', '11'] # sub-30 without data in run-05
        elif level == 4:
            runs = ['04', '06', '09']

        for run in runs:
            df_ecg = pd.read_csv( path+f"/data_ils/cp0{sub_idx}/ecg_0{sub_idx}/cp0{sub_idx}_ecg_level-0{level}B_run-0{run}_dat.csv", index_col=False )
            df_rsp = pd.read_csv(  path+f"/data_ils/cp0{sub_idx}/resp_0{sub_idx}/cp0{sub_idx}_resp_level-0{level}B_run-0{run}_dat.csv", index_col=False )
            # initiate a dataframe for ecg and rsp
            ecg_rsp = pd.DataFrame(data = {"time_dn": df_ecg['time_dn']})
            # Effective sampling rate of sub-03 is 504 Hz
            sampling_rate = 504 if sub_idx == '03' else 128

            for col in ['ecg_projection_ll_ra_mV', 'ecg_projection_la_ra_mV']:
                # clean method see paper (Elgendi, 2010)
                ecg_cleaned = nk.ecg_clean(df_ecg[col],
                                           sampling_rate = sampling_rate,
                                           method = "elgendi2010")
                # Find R-peaks of cleaned channel
                ecg_peaks, info = nk.ecg_peaks(ecg_cleaned,
                                               sampling_rate = sampling_rate,
                                               method = 'elgendi2010',
                                               correct_artifacts = True)
                # quality evaluation see paper (zhao, 2018)
                ecg_quality = nk.ecg_quality(ecg_cleaned,
                                             rpeaks = info["ECG_R_Peaks"],
                                             sampling_rate = sampling_rate,
                                             method = "zhao2018")
                # Find signal rate (per mintue) from peaks
                # Check cleaned ecg quality
                if ecg_quality == "Excellent":
                    ecg_rate = nk.ecg_rate(info, sampling_rate = sampling_rate, desired_length=len(df_ecg[col]))
                    ecg_rsp[col+'_rate'] = ecg_rate
            # respiration signal process, method(Khodadad, 2018)
            rsp_signal, rsp_info = nk.rsp_process(df_rsp['respiration_trace_mV'],
                                                  sampling_rate = sampling_rate,
                                                  method='khodadad2018')
            ecg_rsp['RSP_Rate'] = rsp_signal['RSP_Rate']
            ecg_rsp['RSP_Amplitude'] = rsp_signal['RSP_Amplitude']
            # downsample data in sub-03 to 128 Hz
            if sub_idx == '03':
                idx = np.arange(0, len(df_ecg), 4)
                ecg_rsp = ecg_rsp.iloc[idx].reset_index( drop=True )

            ecg_rsp.to_csv( path+f"/data_ils/cp0{sub_idx}/0{sub_idx}_ecg_rsp/0{sub_idx}_ecg_rsp_level-0{level}_run-0{run}_dat.csv", index=False )

Clean subjects' data file

In [ ]:
# Original dataset contains 35 subjects
# sub-09 is excluded due to missing data in 4 runs
# sub-26 is exlcuded because original signal recordings of multiple runs are low quality
subjects = ['03', '04', '05', '06', '08', 11, 12,
           13, 14, 15, 16, 17, 18, 19, 20,
           22, 23, 24, 25, 27, 28, 29,
           30, 31, 32, 33, 35, 36, 37, 38,
           39, 42, 43]

for sub in subjects:
    clean_ecg_rsp( sub )


# Define a function to compute features from cleaned signals

In [ ]:
def compute_features( df_signal, df_perf, df_ac, sub_id, level, run_id, win_size = 10.0, step = 1 ):
    # first align timestamps
    # convert Matlab time format in each stream to seconds
    df_perf['time_s'] = (df_perf['time_dn'] - df_perf['time_dn'].iloc[0]) * 24 * 3600
    df_signal['time_s'] = (df_signal['time_dn'] - df_signal['time_dn'].iloc[0]) * 24 * 3600
    df_ac['time_s'] = (df_ac['time_dn'] - df_ac['time_dn'].iloc[0]) * 24 * 3600
    # calculate time offset between performance and signal
    offset = (df_perf['time_dn'].iloc[0] - df_signal['time_dn'].iloc[0]) * 24 * 3600
    # align timestamps
    df_perf['time_s_aligned'] = df_perf['time_s'] + offset
    perf_time_aligned = df_perf['time_s_aligned']
    # list to store dictionary of features and targets
    perf_ac_feat = []

    # compute ECG and RSP features per window
    # the first timestamp is non-negative in aligned timestamps
    # Ensure perf_time_aligned is not empty before accessing iloc[0]
    if perf_time_aligned[perf_time_aligned >= 0].empty:
        return pd.DataFrame(perf_ac_feat) # Return empty if no valid performance time

    t = perf_time_aligned[perf_time_aligned >= 0].iloc[0] + win_size
    while t < perf_time_aligned.iloc[-1]:
        t_start = t - win_size
        t_end = t
        # Performance timestamps inside a window
        win_p = (df_perf['time_s_aligned'] > t_start) & (df_perf['time_s_aligned'] <= t_end)
        df_perf_win = df_perf.loc[win_p]
        # Skip if no performance data in window
        if df_perf_win.empty:
            t += step
            continue

        last_perf_row = df_perf_win.iloc[-1]
        # get the metrics inside win_p
        t_aligned = last_perf_row['time_s_aligned']
        error_g = last_perf_row['glideslope_error_deg']
        error_l = last_perf_row['localizer_error_deg']
        error_a = last_perf_row['airspeed_error_kts']
        error_t = last_perf_row['total_error']

        # ECG and RSP window
        win_s = (df_signal['time_s'] > t_start) & (df_signal['time_s'] <= t_end)
        # aircraft timestamps inside a window
        win_ac = (df_ac['time_s'] > t_start) & (df_ac['time_s'] <= t_end)

        # Check if the signal window is empty before computing statistics
        if df_signal.loc[win_s].empty:
            hr_mean = np.nan
            hr_std = np.nan
            hr_min = np.nan
            hr_max = np.nan
            hr_range = np.nan
            br_mean = np.nan
            br_std = np.nan
            amp_mean = np.nan
            amp_std = np.nan
        else:
            if 'ecg_projection_ll_ra_mV_rate' in df_signal.columns:
                ecg_rate_win = df_signal.loc[win_s, 'ecg_projection_ll_ra_mV_rate'].to_numpy()
            else:
                ecg_rate_win = df_signal.loc[win_s, 'ecg_projection_la_ra_mV_rate'].to_numpy()
            rsp_rate_win = df_signal.loc[win_s, 'RSP_Rate'].to_numpy()
            rsp_amp_win = df_signal.loc[win_s, 'RSP_Amplitude'].to_numpy()

            # ECG features (HR)
            hr_mean = float(np.nanmean(ecg_rate_win)) if ecg_rate_win.size > 0 else np.nan
            hr_std = float(np.nanstd(ecg_rate_win)) if ecg_rate_win.size > 0 else np.nan
            hr_min = float(np.nanmin(ecg_rate_win)) if ecg_rate_win.size > 0 else np.nan
            hr_max = float(np.nanmax(ecg_rate_win)) if ecg_rate_win.size > 0 else np.nan
            hr_range = hr_max - hr_min if not np.isnan(hr_max) and not np.isnan(hr_min) else np.nan
            # RSP features (BR, Amplitude)
            br_mean = float(np.nanmean(rsp_rate_win)) if rsp_rate_win.size > 0 else np.nan
            br_std = float(np.nanstd(rsp_rate_win)) if rsp_rate_win.size > 0 else np.nan
            amp_mean = float(np.nanmean(rsp_amp_win)) if rsp_amp_win.size > 0 else np.nan
            amp_std = float(np.nanstd(rsp_amp_win)) if rsp_amp_win.size > 0 else np.nan

            # aircraft states
            if df_ac.loc[win_ac].empty:
                ac_elevation_mean = np.nan
                ac_aileron_mean = np.nan
                ac_gear_mean = np.nan
            else:
                ac_elevation_win = df_ac.loc[win_ac, 'aircraft_elevation_trim'].to_numpy()
                ac_aileron_win = df_ac.loc[win_ac, 'aircraft_aileron_trim'].to_numpy()
                ac_gear_win = df_ac.loc[win_ac, 'aircraft_landing_gear'].to_numpy()
                ac_elevation_mean = float(np.nanmean(ac_elevation_win)) if ac_elevation_win.size > 0 else np.nan
                ac_aileron_mean = float(np.nanmean(ac_aileron_win)) if ac_aileron_win.size > 0 else np.nan
                ac_gear_mean = float(np.nanmean(ac_gear_win)) if ac_gear_win.size > 0 else np.nan

        perf_ac_feat.append({
            'time_s': t_aligned,
            'sub_id': sub_id,
            'level': level,
            'run_id': run_id,
            'glideslope_error_deg': error_g,
            'localizer_error_deg': error_l,
            'airspeed_error_kts': error_a,
            'total_error': error_t,
            'HR_mean': hr_mean,
            'HR_std': hr_std,
            'HR_min': hr_min,
            'HR_max': hr_max,
            'HR_range': hr_range,
            'BR_mean': br_mean,
            'BR_std': br_std,
            'Rsp_amp_mean': amp_mean,
            'Rsp_amp_std': amp_std,
            'elevation_mean': ac_elevation_mean,
            'aileron_mean': ac_aileron_mean,
            'gear_mean': ac_gear_mean
        })
        t += step

    return pd.DataFrame(perf_ac_feat)

compute features from remaining subjects' data and put in a csv file

In [ ]:
# ECG feature extraction
# Remaining 25 subjects after further data quality inspection in cleaned data
# excluded subjects all have abnormal breathing rate in cleaned data
subjects_remain = ['03', '04', '05', '06', '08', 11, 12, 13,
                   15, 16, 17, 18, 19, 20, 22, 23, 24, 27,
                   31, 32, 33, 35, 37, 38, 43]
path = os.getcwd()

for sub_idx in subjects_remain:
    for level in [1, 2, 3, 4]:
        if level == 1:
            runs = ['01', '07', '12']
        elif level == 2:
            runs = ['03', '08', '10']
        elif level == 3:
            runs = ['02', '05', '11']
        elif level == 4:
            runs = ['04', '06', '09']

        for run in runs:
            df_signal = pd.read_csv(path+f"/data_ils/cp0{sub_idx}/0{sub_idx}_ecg_rsp/0{sub_idx}_ecg_rsp_level-0{level}_run-0{run}_dat.csv", index_col=False)
            df_perf = pd.read_csv(path+f"/data_ils/cp0{sub_idx}/perfmetric_0{sub_idx}/cp0{sub_idx}_perfmetric_level-0{level}B_run-0{run}.csv", index_col=False)
            df_ac = pd.read_csv(path+f"/data_ils/cp0{sub_idx}/0{sub_idx}_ac/cp0{sub_idx}_ac_level-0{level}B_run-0{run}_dat.csv", index_col=False)
            df_perf_feat = compute_features( df_signal, df_perf, df_ac, sub_idx, level, run )
            df_perf_feat.to_csv(path+f'/data_ils/cp0{sub_idx}/0{sub_idx}_time_s/0{sub_idx}_time_s_level-0{level}_run-0{run}_dat.csv', index=False)

